In [ ]:
"""METHOD 2, CELL 1: TRAIN QWEN3-32B WITH THE ROUND 1 RECIPE, THEN SCORE IT (G4, about 75 minutes).
A second model family to average with system 2 (3 Gemma models). Same data, same settings, same seed as round 1.
If Colab disconnects: reconnect to a G4 and run THIS cell again. It resumes from the last save (every 100 updates)."""
import sys, gc, json, math, time, uuid, random, shutil, hashlib, subprocess, importlib
from importlib.metadata import version
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
importlib.invalidate_caches()
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel, get_peft_model
from peft.import_utils import is_torchao_available
from project_utils import save_json
from semeval_task3.data import load_dataset

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

DEVICE = "cuda"
MODEL_ID = "Qwen/Qwen3-32B"
SEED = 2027
RUN = PROJECT / "outputs/runs/step18_qwen3_32b"
R1_DATA = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
ENS_DECISION = PROJECT / "outputs/runs/step15_seed_ensemble/ensemble_decision.json"
EFFECTIVE_BATCH, LR, SAVE_EVERY = 16, 1e-4, 100
SYSTEM = "You are an expert lexicographer of historical Swedish."

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

# ---------------------------------------------------------------------------
section("0. THE RULE (saved before any training)")
RUN.mkdir(parents=True, exist_ok=True)
ENS_SV = json.loads(ENS_DECISION.read_text())["swedish"]["ensemble (average of 3)"]
RULE = dict(
    new_system="score = average of (system 2 score, Qwen score); both are Yes-minus-No log-odds, so they share units; one cutoff fitted on Swedish",
    health="Qwen: loss went down, all scores finite, Qwen alone Swedish macro-F1 >= 0.6818 (plain untrained Gemma level), predicts both Yes and No",
    rule_1a=f"new system Swedish macro-F1 (nested) >= {ENS_SV + 0.02:.4f} (system 2 {ENS_SV:.4f} + 0.02)",
    rule_1b="better than system 2 on at least 6 of 10 Swedish words",
    rule_2="no other language (EN, RU, IT, ES, NL) loses more than 0.02 AUC or 0.02 macro-F1 vs system 2",
    if_all_pass="new system (3 Gemma + Qwen) becomes the main system",
    if_any_fail="drop Qwen; system 2 stays the main system")
if (RUN / "rule.json").exists():
    assert json.loads((RUN / "rule.json").read_text()) == RULE, "The saved rule differs from this cell: STOP and paste this output."
    print("Rule already saved earlier and unchanged.")
else:
    save_json(RUN / "rule.json", RULE)
for k, v in RULE.items():
    print(f"{k:<12}: {v}")

# ---------------------------------------------------------------------------
section("1. CHECK EVERY INPUT FILE")
is_torchao_available.cache_clear()
assert is_torchao_available() is False, "torchao is still visible to peft: Runtime > Restart session, then run this cell again."
m1 = json.loads((R1_DATA / "manifest.json").read_text())
train_path = R1_DATA / "pilot_train.jsonl"
assert digest(train_path) == m1["sha256"]["pilot_train.jsonl"], "Round 1 training file changed: STOP."
training = read_rows(train_path)
assert len(training) == 8510
for r in training:
    assert int(r["label"]) in (0, 1) and r["messages"][-1]["role"] == "user"
    assert all(m["role"] in ("system", "user") for m in r["messages"])
enru_manifest = json.loads((ENRU_EVAL / "manifest.json").read_text())
xl_manifest = json.loads((XL_EVAL / "manifest.json").read_text())
eval_sets = {}
for name in ("EN_wic_dev", "RU_axolotl_test_derived"):
    assert digest(ENRU_EVAL / f"{name}.jsonl") == enru_manifest["output_sha256"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(ENRU_EVAL / f"{name}.jsonl")
for lang in ("IT", "ES", "NL"):
    name = f"{lang}_xlwsd_test"
    assert digest(XL_EVAL / f"{name}.jsonl") == xl_manifest["files"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(XL_EVAL / f"{name}.jsonl")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=False)
eval_sets["SV_dev"] = [dict(id="|".join(u.key), messages=[
    dict(role="system", content=SYSTEM),
    dict(role="user", content=(f"Target word (lemma): {u.word}\n"
                               f"Sense definition (Swedish): {u.definition}\n"
                               f"Sentence from {u.year}: {u.marked_text()}\n\n"
                               "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
                               "Answer only Yes or No."))]) for u in sv.usages]
assert len(eval_sets["SV_dev"]) == 1983
print(f"Round 1 training file : hash PASS | {len(training):,} examples (the exact data round 1 Gemma trained on)")
print(f"Evaluation sets       : hash PASS | { {k: len(v) for k, v in eval_sets.items()} }")
TOTAL = math.ceil(len(training) / EFFECTIVE_BATCH)
WARMUP = math.ceil(0.05 * TOTAL)
print(f"Training: {TOTAL} updates (one pass), warm-up {WARMUP}, seed {SEED}, final update is used (fixed in advance)")

# ---------------------------------------------------------------------------
section("2. TOKENIZER CHECK (seconds, before the big download)")
qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
qwen_tokenizer.padding_side = "left"
assert qwen_tokenizer.pad_token is not None, "Qwen tokenizer has no padding token: STOP and paste this output."
def render(messages):
    return qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
example = render(training[0]["messages"])
print("End of one prompt as Qwen sees it:", repr(example[-60:]))
assert example.endswith("</think>\n\n"), "Thinking mode is not switched off in the prompt: STOP and paste this output."
YES = qwen_tokenizer.encode("Yes", add_special_tokens=False)
NO = qwen_tokenizer.encode("No", add_special_tokens=False)
assert len(YES) == 1 and len(NO) == 1 and YES != NO, f"Yes/No are not single tokens ({YES}, {NO}): STOP and paste this output."
YES_ID, NO_ID = YES[0], NO[0]
print(f"Thinking mode off: PASS | 'Yes' = token {YES_ID}, 'No' = token {NO_ID}: PASS | transformers {version('transformers')}")

# ---------------------------------------------------------------------------
section("3. LOAD QWEN3-32B (bf16; first time downloads about 65 GB, about 10 minutes)")
if "qwen_model" in globals() or "full_model" in globals() or "BASE" in globals():
    print("Qwen is already loaded in this session: reusing it.")
else:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 2**30:.0f} GiB")
    assert props.total_memory / 2**30 > 70, "Qwen 32B training needs a GPU with more than 70 GB: use the G4."
    qwen_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
    print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
QWEN_REVISION = getattr((globals().get("qwen_model") or globals().get("BASE") or globals().get("full_model")).config, "_commit_hash", None)
print(f"Qwen revision used: {QWEN_REVISION}")

def plain_base():
    """Qwen with every adapter removed (unload() removes adapters without merging them)."""
    current = None
    for name in ("full_model", "BASE", "qwen_model"):
        if globals().get(name) is not None:
            current = globals()[name]
            break
    base = current.unload() if isinstance(current, PeftModel) else current
    for name in ("full_model", "qwen_model"):
        globals().pop(name, None)
    assert not any("lora_" in n for n, _ in base.named_parameters())
    gc.collect()
    torch.cuda.empty_cache()
    return base

def encode(batch):
    inputs = qwen_tokenizer([render(r["messages"]) for r in batch], padding=True, return_tensors="pt",
                            add_special_tokens=False).to(DEVICE)
    assert inputs["attention_mask"][:, -1].all(), "Padding is not on the left."
    return inputs

def score_rows(model, rows, destination, tag):
    """Resumable scoring: labels never enter the model; saved to Drive every 100 batches."""
    identity = hashlib.sha256(json.dumps([(str(r["id"]), r["messages"]) for r in rows],
                                         ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    if destination.exists():
        saved = json.loads(destination.read_text())
        assert saved["input_sha256"] == identity and saved["model_tag"] == tag, f"{destination.name}: saved scores belong to other inputs."
    else:
        saved = dict(input_sha256=identity, model_tag=tag, scores={})
    pending = [r for r in rows if str(r["id"]) not in saved["scores"]]
    model.eval()
    t0 = time.perf_counter()
    try:
        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16):
            for b, i in enumerate(range(0, len(pending), 2), 1):
                batch = pending[i:i + 2]
                logits = model(**encode(batch), use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
                assert torch.isfinite(logits).all()
                for r, margin in zip(batch, (logits[:, YES_ID] - logits[:, NO_ID]).tolist()):
                    saved["scores"][str(r["id"])] = dict(margin=float(margin))
                del logits
                if b % 100 == 0:
                    save_json(destination, saved)
    finally:
        save_json(destination, saved)
    assert set(saved["scores"]) == {str(r["id"]) for r in rows}
    if pending:
        print(f"      {tag}: {len(rows):,} scores ({(time.perf_counter() - t0) / 60:.1f} min)", flush=True)
    return saved

# ---------------------------------------------------------------------------
section("4. TRAIN QWEN WITH THE ROUND 1 RECIPE (about 50 minutes)")
config = dict(model=MODEL_ID, revision=QWEN_REVISION, seed=SEED, recipe="round 1 (step09) unchanged, applied to Qwen3-32B",
              examples=len(training), train_sha256=digest(train_path), micro_batch=1, effective_batch=EFFECTIVE_BATCH,
              learning_rate=LR, warmup_steps=WARMUP, steps=TOTAL, schedule="linear warmup then linear decay",
              r=16, alpha=32, dropout=0.05, target="all attention q/k/v/o", base_precision="BF16, no quantization",
              thinking="off", yes_token=YES_ID, no_token=NO_ID, checkpoint="final update (fixed before training)")
if (RUN / "config.json").exists():
    assert json.loads((RUN / "config.json").read_text()) == json.loads(json.dumps(config)), \
        "Settings changed since this run started: STOP and paste this message."
else:
    save_json(RUN / "config.json", config)

latest_file, final_dir = RUN / "latest.json", RUN / "adapter_final"
if (RUN / "COMPLETE.json").exists():
    print("Qwen run already complete: nothing to do.")
else:
    BASE = plain_base()
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); random.seed(SEED); np.random.seed(SEED)
    order = list(training)
    random.Random(SEED).shuffle(order)
    n_layers = BASE.config.num_hidden_layers
    targets = [n for n, m in BASE.named_modules() if isinstance(m, torch.nn.Linear)
               and n.split(".")[-1] in ("q_proj", "k_proj", "v_proj", "o_proj")]
    assert len(targets) == 4 * n_layers, f"Expected {4 * n_layers} attention layers, found {len(targets)}: STOP and paste this output."
    print(f"LoRA on {len(targets)} attention layers ({n_layers} blocks x q/k/v/o)")
    state = None
    if (final_dir / "adapter_config.json").exists():
        full_model = PeftModel.from_pretrained(BASE, str(final_dir), is_trainable=False)
        step_done, log = TOTAL, json.loads((RUN / "training_log.json").read_text())
        print("Training already finished; continuing with scoring.")
    elif latest_file.exists():
        latest = Path(json.loads(latest_file.read_text())["directory"])
        full_model = PeftModel.from_pretrained(BASE, str(latest / "adapter"), is_trainable=True)
        state = torch.load(latest / "training_state.pt", map_location="cpu", weights_only=False)     # our own file
        step_done, log = state["step"], state["log"]
        print(f"Resuming after update {step_done} of {TOTAL}.")
    else:
        full_model = get_peft_model(BASE, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=targets, bias="none"))
        step_done, log = 0, []
        assert all(torch.count_nonzero(p).item() == 0 for n, p in full_model.named_parameters() if "lora_B" in n)
        trainable = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
        print(f"Fresh adapter attached (starts from zero) | {trainable / 1e6:.1f}M trainable parameters")

    if step_done < TOTAL:
        full_model.config.use_cache = False
        full_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        params = [p for p in full_model.parameters() if p.requires_grad]
        assert params and all("lora_" in n for n, p in full_model.named_parameters() if p.requires_grad)
        optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
        if state:
            optimizer.load_state_dict(state["optimizer"])
            random.setstate(state["python_rng"]); np.random.set_state(state["numpy_rng"]); torch.set_rng_state(state["torch_rng"])
            if state.get("cuda_rng") is not None:
                torch.cuda.set_rng_state_all(state["cuda_rng"])
        del state

        def save_resume(step):
            folder = RUN / f"resume_{step:04d}_{uuid.uuid4().hex[:8]}"
            folder.mkdir()
            full_model.save_pretrained(folder / "adapter", safe_serialization=True)
            torch.save(dict(step=step, log=log, optimizer=optimizer.state_dict(), python_rng=random.getstate(),
                            numpy_rng=np.random.get_state(), torch_rng=torch.get_rng_state(),
                            cuda_rng=torch.cuda.get_rng_state_all()), folder / "training_state.pt")
            previous = json.loads(latest_file.read_text())["directory"] if latest_file.exists() else None
            save_json(latest_file, {"directory": str(folder), "step": step})
            if previous and Path(previous).parent == RUN and Path(previous).name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            save_json(RUN / "training_log.json", log)

        full_model.train()
        t0 = time.perf_counter()
        for step in range(step_done + 1, TOTAL + 1):
            batch_rows = order[(step - 1) * EFFECTIVE_BATCH:step * EFFECTIVE_BATCH]
            scale = step / WARMUP if step <= WARMUP else (TOTAL - step + 1) / (TOTAL - WARMUP)
            for pg in optimizer.param_groups:
                pg["lr"] = LR * scale
            optimizer.zero_grad(set_to_none=True)
            loss_sum, correct = 0.0, 0
            for r in batch_rows:
                answer = torch.tensor([YES_ID if int(r["label"]) else NO_ID], device=DEVICE)
                with torch.autocast(DEVICE, dtype=torch.bfloat16):
                    logits = full_model(**encode([r]), use_cache=False, logits_to_keep=1).logits[:, -1, :]
                loss = F.cross_entropy(logits.float(), answer)
                assert torch.isfinite(loss), "Non-finite loss: STOP and paste this message. The last saved checkpoint is intact."
                (loss / len(batch_rows)).backward()
                loss_sum += float(loss.detach())
                correct += int((logits[0, YES_ID] > logits[0, NO_ID]).item() == bool(int(r["label"])))
                del logits, loss, answer
            norm = float(torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True))
            optimizer.step()
            log.append(dict(step=step, loss=loss_sum / len(batch_rows), batch_accuracy=correct / len(batch_rows),
                            grad_norm=norm, lr=LR * scale))
            if step == 1 or step % 50 == 0 or step == TOTAL:
                minutes = (time.perf_counter() - t0) / 60
                print(f"   qwen | update {step:>3}/{TOTAL}: loss {loss_sum / len(batch_rows):.4f} | accuracy {correct / len(batch_rows):.2f} | "
                      f"about {minutes / (step - step_done) * (TOTAL - step):.0f} min of training left", flush=True)
            if step % SAVE_EVERY == 0 and step < TOTAL:
                save_resume(step)
        full_model.save_pretrained(final_dir, safe_serialization=True)
        save_json(RUN / "training_log.json", log)
        optimizer.zero_grad(set_to_none=True)
        del optimizer, params
        if latest_file.exists():
            previous = Path(json.loads(latest_file.read_text())["directory"])
            if previous.parent == RUN and previous.name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            latest_file.unlink()
        gc.collect(); torch.cuda.empty_cache()
        losses = [x["loss"] for x in log]
        print(f"Training done | mean loss first 50 updates {np.mean(losses[:50]):.3f} -> last 50 updates {np.mean(losses[-50:]):.3f}")

    # -----------------------------------------------------------------------
    section("5. SCORE QWEN ON SWEDISH + THE 5 OTHER-LANGUAGE TEST SETS (about 15 minutes)")
    (RUN / "eval_scores").mkdir(exist_ok=True)
    for name, rows_to_score in eval_sets.items():
        score_rows(full_model, rows_to_score, RUN / "eval_scores" / f"{name}.json", f"qwen_{name}")
    save_json(RUN / "COMPLETE.json", dict(model=MODEL_ID, revision=QWEN_REVISION, adapter=str(final_dir),
                                          adapter_sha256=digest(final_dir / "adapter_model.safetensors"),
                                          eval_scores={name: f"eval_scores/{name}.json" for name in eval_sets}))
    print("Qwen: COMPLETE")

section("SUMMARY")
print(f"Qwen run: {'COMPLETE' if (RUN / 'COMPLETE.json').exists() else 'NOT complete'} | saved in {RUN}")
print("System 1 and system 2 are untouched. Nothing is decided yet.")
print("Next: paste this output, then run the decision cell (no GPU needed).")

In [ ]:
"""METHOD 2, CELL 2: THE DECISION (no GPU work, about 2 minutes).
Shows Qwen alone, system 2, and their average, then applies the rule saved before training."""
import sys, json, hashlib, subprocess, importlib.util
from collections import defaultdict
from pathlib import Path
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from sklearn.metrics import roc_auc_score
from project_utils import save_json, frozen
from semeval_task3.data import load_dataset
import semeval_task3.cv as cv
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score as positive_score, word_bootstrap

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

RUN = PROJECT / "outputs/runs/step18_qwen3_32b"
ENS_ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
R1_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
XL_SCORES = PROJECT / "outputs/runs/step11_xlwsd_scores/adapter_6f3d236ac50e"
SCORER_DIR, SCORER_COMMIT = Path("/content/official-scorer-latest"), "0e69238"
GRID = tuple([-1000.0] + np.round(np.arange(-100, 100.001, 0.25), 2).tolist() + [1000.0])
EXTERNAL = ("EN_wic_dev", "RU_axolotl_test_derived", "IT_xlwsd_test", "ES_xlwsd_test", "NL_xlwsd_test")
PLAIN_GEMMA_SV = 0.6818
S2, QW, MIX = "system 2 (3 Gemma)", "Qwen alone", "Qwen + system 2"

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def macro_score(gold, predictions):
    """Official metric: per word, the mean of Yes-F1 and No-F1; then the mean over words."""
    report = positive_score(gold, predictions)
    languages = defaultdict(list)
    for row in report["per_word"]:
        tp, fp, fn = row["tp"], row["fp"], row["fn"]
        tn = row["n"] - tp - fp - fn
        parts = []
        if tp + fp + fn > 0:
            parts.append(2 * tp / (2 * tp + fp + fn))
        if tn + fp + fn > 0:
            parts.append(2 * tn / (2 * tn + fn + fp))
        row["positive_f1"], row["f1"] = row["f1"], sum(parts) / len(parts)
        languages[row["language"]].append(row["f1"])
    report["per_language"] = {k: sum(v) / len(v) for k, v in languages.items()}
    report["average"] = sum(report["per_language"].values()) / len(report["per_language"])
    return report

def pooled_macro(y, s, t):
    y, p = np.asarray(y), (np.asarray(s) >= t).astype(int)
    f = lambda c: 2 * np.sum((y == c) & (p == c)) / max(np.sum(y == c) + np.sum(p == c), 1)
    return float((f(1) + f(0)) / 2)

# ---------------------------------------------------------------------------
section("1. LOAD THE RULE AND THE SCORES")
assert (RUN / "COMPLETE.json").exists(), "The Qwen run is not finished: run cell 1 until its SUMMARY says COMPLETE."
RULE = json.loads((RUN / "rule.json").read_text())
ENS_SV = json.loads((ENS_ROOT / "ensemble_decision.json").read_text())["swedish"]["ensemble (average of 3)"]
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv.require_gold()
keys = sorted(gold)
def by_key(saved, tag):
    assert saved["model_tag"] == tag, f"Unexpected scores: {saved['model_tag']}"
    return {u.key: float(saved["scores"]["|".join(u.key)]["margin"]) for u in sv.usages}
gemma = [by_key(json.loads((R1_RUN / "SV_scores.json").read_text()), "chosen_532_SV")] + \
        [by_key(json.loads((ENS_ROOT / f"seed_{s}/eval_scores/SV_dev.json").read_text()), f"seed_{s}_SV_dev") for s in (2028, 2029)]
sys2 = {k: float(np.mean([g[k] for g in gemma])) for k in keys}
qwen = by_key(json.loads((RUN / "eval_scores/SV_dev.json").read_text()), "qwen_SV_dev")
mix = {k: (sys2[k] + qwen[k]) / 2 for k in keys}
scores = {S2: sys2, QW: qwen, MIX: mix}
for name, s in scores.items():
    v = np.array(list(s.values()))
    assert np.isfinite(v).all() and np.abs(v).max() < 100, f"{name}: scores outside the cutoff grid: STOP."
r = float(np.corrcoef([sys2[k] for k in keys], [qwen[k] for k in keys])[0, 1])
print("The new-system rule, saved before training:")
for k, v in RULE.items():
    print(f"   {k:<12}: {v}")
print(f"\nHow similar Qwen and system 2 are on Swedish (correlation): {r:.3f}")
print("   (the 3 Gemma seeds were 0.96 to 0.98 with each other; lower = more room for averaging to help)")

# ---------------------------------------------------------------------------
if not SCORER_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/ChangeIsKey/semeval2027-task3.git", str(SCORER_DIR)], check=True)
subprocess.run(["git", "-C", str(SCORER_DIR), "checkout", "-q", SCORER_COMMIT], check=True)
assert "average='macro'" in (SCORER_DIR / "evaluation.py").read_text()
sys.path.insert(0, str(SCORER_DIR))
spec = importlib.util.spec_from_file_location("official_eval_latest", SCORER_DIR / "evaluation.py")
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
def official_check(predictions):
    return official.compute_f1_score({f"{k[1]};{k[2]}": int(v) for k, v in gold.items()},
                                     {f"{k[1]};{k[2]}": int(predictions[k]) for k in gold}, "check")

section("2. SWEDISH DEV, OFFICIAL METRIC (nested leave-one-word-out, same procedure as before)")
def nested(s, name):
    cv.score = macro_score
    try:
        return nested_word_cv(sv, [Candidate(name, frozen(s), GRID)])
    finally:
        cv.score = positive_score
res = {name: nested(s, f"c{i}") for i, (name, s) in enumerate(scores.items())}
for name in res:
    assert abs(official_check(res[name]["predictions"]) - res[name]["report"]["average"]) < 1e-12, f"{name}: official scorer disagrees: STOP."
assert abs(res[S2]["report"]["average"] - ENS_SV) < 1e-9, "System 2 no longer reproduces 0.7664: STOP and paste this output."
print(f"System 2 reproduces {ENS_SV:.4f}: PASS | official scorer agrees on all three: PASS\n")
for name in res:
    print(f"   {name:<20} macro-F1 {res[name]['report']['average']:.4f}")
per = {name: {row["word"]: row["f1"] for row in res[name]["report"]["per_word"]} for name in res}
words = sorted(per[S2])
better = sum(per[MIX][w] > per[S2][w] + 1e-9 for w in words)
worse = sum(per[MIX][w] < per[S2][w] - 1e-9 for w in words)
print(f"\n{'word':<11}{'system 2':>10}{'Qwen':>8}{'mix':>8}{'mix minus sys 2':>17}")
for w in words:
    print(f"{w:<11}{per[S2][w]:>10.3f}{per[QW][w]:>8.3f}{per[MIX][w]:>8.3f}{per[MIX][w] - per[S2][w]:>+17.3f}")
paired = word_bootstrap(res[MIX]["report"], other=res[S2]["report"])
print(f"\nMix minus system 2: {paired['estimate']:+.4f} (95% range {paired['lower']:+.4f} to {paired['upper']:+.4f}) | "
      f"better on {better}, worse on {worse}, same on {10 - better - worse} of 10 words")

def fitted(s):
    v = np.array([s[k] for k in keys])
    t = max(GRID, key=lambda x: (macro_score(gold, dict(zip(keys, (v >= x).astype(int).tolist())))["average"], -x))
    return float(t), macro_score(gold, dict(zip(keys, (v >= t).astype(int).tolist())))["average"]
cut = {name: fitted(s) for name, s in scores.items()}
print("Cutoffs fitted on all 10 words (used for the other languages): " + " | ".join(f"{n} {c[0]:+.2f}" for n, c in cut.items()))

# ---------------------------------------------------------------------------
section("3. HEALTH CHECKS FOR QWEN (it is dropped if any fails, whatever the mix scores)")
log = json.loads((RUN / "training_log.json").read_text())
cfg = json.loads((RUN / "config.json").read_text())
losses = [x["loss"] for x in log]
first, last = float(np.mean(losses[:50])), float(np.mean(losses[-50:]))
q_pred = [qwen[k] >= cut[QW][0] for k in keys]
health = {
    f"all {cfg['steps']} updates in the log": len(log) == cfg["steps"],
    f"loss went down ({first:.3f} -> {last:.3f})": last < first,
    f"Qwen alone >= {PLAIN_GEMMA_SV} on Swedish (plain Gemma level): {res[QW]['report']['average']:.4f}": res[QW]["report"]["average"] >= PLAIN_GEMMA_SV,
    f"Qwen predicts both Yes and No ({sum(q_pred):,} Yes of {len(q_pred):,})": 0 < sum(q_pred) < len(q_pred),
}
for name, ok in health.items():
    print(f"   {'PASS' if ok else 'FAIL'}  {name}")
health_ok = all(health.values())

# ---------------------------------------------------------------------------
section("4. THE OTHER FIVE LANGUAGES (AUC and macro-F1; the rule compares the mix with system 2)")
ext, all_ok = {}, True
print(f"{'data':<26}{'AUC s2':>8}{'AUC Qw':>8}{'AUC mix':>9}{'change':>8}   {'mF1 s2':>8}{'mF1 Qw':>8}{'mF1 mix':>9}{'change':>8}  rule")
for name in EXTERNAL:
    rows = read_jsonl((ENRU_EVAL if name[:2] in ("EN", "RU") else XL_EVAL) / f"{name}.jsonl")
    ids, y = [str(row["id"]) for row in rows], [int(row["label"]) for row in rows]
    if name[:2] in ("EN", "RU"):
        r1 = json.loads((R1_RUN / f"{name}_scores.json").read_text())["scores"]
        r1 = [r1[i]["margin"] for i in ids]
    else:
        r1 = json.loads((XL_SCORES / f"{name[:2]}_trained_scores.json").read_text())
        r1 = [r1[i] for i in ids]
    g = [r1] + [[json.loads((ENS_ROOT / f"seed_{s}/eval_scores/{name}.json").read_text())["scores"][i]["margin"] for i in ids]
                for s in (2028, 2029)]
    s2 = np.mean(g, axis=0)
    qsaved = json.loads((RUN / f"eval_scores/{name}.json").read_text())
    assert qsaved["model_tag"] == f"qwen_{name}", f"Unexpected scores: {qsaved['model_tag']}"
    q = np.array([qsaved["scores"][i]["margin"] for i in ids])
    assert np.isfinite(q).all(), f"Qwen {name}: non-finite scores: STOP."
    m = (s2 + q) / 2
    a = {n: float(roc_auc_score(y, s)) for n, s in ((S2, s2), (QW, q), (MIX, m))}
    f = {n: pooled_macro(y, s, cut[n][0]) for n, s in ((S2, s2), (QW, q), (MIX, m))}
    ok = bool(a[MIX] - a[S2] >= -0.02 and f[MIX] - f[S2] >= -0.02)
    all_ok &= ok
    ext[name] = dict(auc=a, macro_f1=f, rule_pass=ok)
    print(f"{name:<26}{a[S2]:>8.4f}{a[QW]:>8.4f}{a[MIX]:>9.4f}{a[MIX] - a[S2]:>+8.4f}   "
          f"{f[S2]:>8.4f}{f[QW]:>8.4f}{f[MIX]:>9.4f}{f[MIX] - f[S2]:>+8.4f}  {'PASS' if ok else 'FAIL'}")
print("Each system uses its own Swedish cutoff. External sets are not competition data.")

# ---------------------------------------------------------------------------
section("5. THE RULE (saved before training)")
mix_sv = res[MIX]["report"]["average"]
r1a, r1b = mix_sv >= ENS_SV + 0.02, better >= 6
print(f"Health   Qwen passes every health check                    : {'PASS' if health_ok else 'FAIL'}")
print(f"Rule 1a  mix Swedish macro-F1 >= {ENS_SV + 0.02:.4f} (system 2 + 0.02)  : {mix_sv:.4f} -> {'PASS' if r1a else 'FAIL'}")
print(f"Rule 1b  mix better than system 2 on >= 6 of 10 words      : {better}/10   -> {'PASS' if r1b else 'FAIL'}")
print(f"Rule 2   no other language loses > 0.02 AUC or macro-F1    : {'PASS' if all_ok else 'FAIL'}")
adopt = bool(health_ok and r1a and r1b and all_ok)
save_json(RUN / "qwen_decision.json", dict(
    metric="per-word macro-F1, official scorer commit 0e69238, nested leave-one-word-out",
    swedish={n: res[n]["report"]["average"] for n in res}, correlation_qwen_vs_system2=r, per_word=per,
    paired_mix_minus_system2=paired, words_better=int(better), words_worse=int(worse),
    cutoffs={n: dict(cutoff=c[0], in_sample=c[1]) for n, c in cut.items()},
    health={k: bool(v) for k, v in health.items()}, external=ext,
    rules=dict(health=health_ok, r1a=bool(r1a), r1b=bool(r1b), r2=bool(all_ok)), adopt=adopt,
    note="Records the decision only. Nothing is frozen or replaced by this cell."))

section("DECISION")
if adopt:
    print(f"ADOPT THE MIX: Swedish {ENS_SV:.4f} -> {mix_sv:.4f}, no language worse, Qwen healthy.")
    print("Next: a freeze cell for the new main system (3 Gemma + Qwen, cutoff from section 2).")
else:
    print(f"DROP QWEN. The mix scored {mix_sv:.4f} vs system 2 {ENS_SV:.4f} (Qwen alone {res[QW]['report']['average']:.4f}).")
    print("System 2 stays the main system, exactly as it is.")
print("Nothing was frozen or changed. Saved:", RUN / "qwen_decision.json")